# Complete Meta-Analysis Workflow

This notebook demonstrates a complete meta-analysis workflow using ASReview 5-Star.

## Workflow Steps
1. Prepare study data
2. Pool effects (fixed and random)
3. Assess heterogeneity
4. Test for publication bias
5. Generate forest plot data
6. Sensitivity analysis

In [ ]:
import asreview_5star as a5s
import math
import numpy as np
import matplotlib.pyplot as plt

## Step 1: Prepare Study Data

Example: Meta-analysis of treatment effect on mortality (Hazard Ratios)

In [ ]:
# Study data from included RCTs
studies = [
    {"name": "PARADIGM-HF (2014)", "hr": 0.80, "ci_lower": 0.73, "ci_upper": 0.87, "n": 8442},
    {"name": "DAPA-HF (2019)", "hr": 0.82, "ci_lower": 0.73, "ci_upper": 0.92, "n": 4744},
    {"name": "EMPEROR-Reduced (2020)", "hr": 0.76, "ci_lower": 0.62, "ci_upper": 0.90, "n": 3730},
    {"name": "GALACTIC-HF (2021)", "hr": 0.91, "ci_lower": 0.82, "ci_upper": 1.01, "n": 8256},
    {"name": "VICTORIA (2020)", "hr": 0.90, "ci_lower": 0.82, "ci_upper": 0.98, "n": 5050},
    {"name": "DELIVER (2022)", "hr": 0.90, "ci_lower": 0.78, "ci_upper": 1.04, "n": 6263},
    {"name": "STRONG-HF (2022)", "hr": 0.66, "ci_lower": 0.50, "ci_upper": 0.86, "n": 1078},
]

# Convert to log scale for analysis
def hr_to_log(hr, ci_lower, ci_upper):
    log_hr = math.log(hr)
    se = (math.log(ci_upper) - math.log(ci_lower)) / 3.92  # 1.96 * 2
    return log_hr, se

effects = []
ses = []
names = []
sample_sizes = []

for study in studies:
    log_hr, se = hr_to_log(study["hr"], study["ci_lower"], study["ci_upper"])
    effects.append(log_hr)
    ses.append(se)
    names.append(study["name"])
    sample_sizes.append(study["n"])

print("Prepared data for", len(studies), "studies")
print(f"Total sample size: {sum(sample_sizes):,}")

## Step 2: Pool Effects

In [ ]:
# Fixed effects model
fixed_result = a5s.pool_effects(effects, ses, study_names=names, model="fixed")

# Random effects model
random_result = a5s.pool_effects(effects, ses, study_names=names, model="random")

print("=" * 60)
print("FIXED EFFECTS MODEL")
print("=" * 60)
print(f"Pooled HR: {math.exp(fixed_result.pooled_effect):.3f}")
print(f"95% CI: [{math.exp(fixed_result.ci_lower):.3f}, {math.exp(fixed_result.ci_upper):.3f}]")
print(f"P-value: {fixed_result.p_value:.6f}")
print(f"Z-score: {fixed_result.z_score:.3f}")

print("\n" + "=" * 60)
print("RANDOM EFFECTS MODEL")
print("=" * 60)
print(f"Pooled HR: {math.exp(random_result.pooled_effect):.3f}")
print(f"95% CI: [{math.exp(random_result.ci_lower):.3f}, {math.exp(random_result.ci_upper):.3f}]")
print(f"P-value: {random_result.p_value:.6f}")
print(f"Z-score: {random_result.z_score:.3f}")

## Step 3: Assess Heterogeneity

In [ ]:
het = random_result.heterogeneity

print("HETEROGENEITY ASSESSMENT")
print("=" * 60)
print(f"Cochran's Q: {het['Q']:.2f} (df={het['df']}, p={het['Q_p_value']:.4f})")
print(f"I-squared: {het['I_squared']:.1f}%")
print(f"H-squared: {het['H_squared']:.2f}")
print(f"Tau-squared: {het['tau_squared']:.4f}")
print(f"Tau: {het['tau']:.4f}")

# Interpretation
if het['I_squared'] < 25:
    interpretation = "Low heterogeneity"
elif het['I_squared'] < 50:
    interpretation = "Moderate heterogeneity"
elif het['I_squared'] < 75:
    interpretation = "Substantial heterogeneity"
else:
    interpretation = "Considerable heterogeneity"

print(f"\nInterpretation: {interpretation}")

# Prediction interval
pi = het.get('prediction_interval', {})
if pi:
    print(f"\n95% Prediction Interval (HR): [{math.exp(pi['lower']):.3f}, {math.exp(pi['upper']):.3f}]")

## Step 4: Test for Publication Bias

In [ ]:
# Egger's test (regression-based)
egger = a5s.eggers_test(effects, ses)

# Begg's test (rank correlation)
begg = a5s.beggs_test(effects, ses)

print("PUBLICATION BIAS TESTS")
print("=" * 60)
print(f"\nEgger's Test:")
print(f"  Intercept: {egger.test_statistic:.3f}")
print(f"  P-value: {egger.p_value:.4f}")
print(f"  Interpretation: {egger.interpretation}")

print(f"\nBegg's Test:")
print(f"  Kendall's Tau: {begg.details.get('kendall_tau', 'N/A'):.3f}")
print(f"  P-value: {begg.p_value:.4f}")
print(f"  Interpretation: {begg.interpretation}")

## Step 5: Generate Forest Plot Data

In [ ]:
from asreview_5star import forest_plot_data

# Get forest plot data with exponentiation (for HR display)
plot_data = forest_plot_data(effects, ses, names, exponentiate=True)

print("FOREST PLOT DATA")
print("=" * 60)
print(f"{'Study':<30} {'HR':<8} {'95% CI':<20} {'Weight':<8}")
print("-" * 70)

for study in plot_data['studies']:
    ci_str = f"[{study['ci_lower']:.3f}, {study['ci_upper']:.3f}]"
    print(f"{study['name']:<30} {study['effect']:.3f}   {ci_str:<20} {study['weight']:.1f}%")

print("-" * 70)
pooled = plot_data['pooled']
ci_str = f"[{pooled['ci_lower']:.3f}, {pooled['ci_upper']:.3f}]"
print(f"{'POOLED (Random Effects)':<30} {pooled['effect']:.3f}   {ci_str:<20}")

In [ ]:
# Visualize as a simple forest plot
fig, ax = plt.subplots(figsize=(10, 8))

studies_data = plot_data['studies']
y_positions = range(len(studies_data) + 1)  # +1 for pooled

# Plot individual studies
for i, study in enumerate(studies_data):
    y = len(studies_data) - i
    ax.errorbar(study['effect'], y, 
                xerr=[[study['effect'] - study['ci_lower']], [study['ci_upper'] - study['effect']]],
                fmt='s', markersize=8, capsize=3, color='navy')
    ax.text(-0.05, y, study['name'], ha='right', va='center', fontsize=9)
    ax.text(1.35, y, f"{study['effect']:.2f} [{study['ci_lower']:.2f}, {study['ci_upper']:.2f}]", 
            ha='left', va='center', fontsize=8)

# Plot pooled estimate
pooled = plot_data['pooled']
ax.errorbar(pooled['effect'], 0, 
            xerr=[[pooled['effect'] - pooled['ci_lower']], [pooled['ci_upper'] - pooled['effect']]],
            fmt='D', markersize=10, capsize=5, color='darkred')
ax.text(-0.05, 0, 'Pooled (Random)', ha='right', va='center', fontsize=9, fontweight='bold')
ax.text(1.35, 0, f"{pooled['effect']:.2f} [{pooled['ci_lower']:.2f}, {pooled['ci_upper']:.2f}]", 
        ha='left', va='center', fontsize=8, fontweight='bold')

# Reference line at HR = 1
ax.axvline(x=1, color='gray', linestyle='--', alpha=0.7)

ax.set_xlim(0.4, 1.3)
ax.set_ylim(-1, len(studies_data) + 1)
ax.set_xlabel('Hazard Ratio', fontsize=11)
ax.set_title('Forest Plot: Treatment Effect on Mortality', fontsize=12, fontweight='bold')
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

plt.tight_layout()
plt.show()

## Step 6: Sensitivity Analysis (Leave-One-Out)

In [ ]:
print("LEAVE-ONE-OUT SENSITIVITY ANALYSIS")
print("=" * 60)
print(f"{'Excluded Study':<30} {'Pooled HR':<12} {'95% CI':<20}")
print("-" * 62)

sensitivity_results = []

for i in range(len(effects)):
    # Exclude study i
    loo_effects = effects[:i] + effects[i+1:]
    loo_ses = ses[:i] + ses[i+1:]
    loo_names = names[:i] + names[i+1:]
    
    result = a5s.pool_effects(loo_effects, loo_ses, model="random")
    hr = math.exp(result.pooled_effect)
    ci_l = math.exp(result.ci_lower)
    ci_u = math.exp(result.ci_upper)
    
    sensitivity_results.append({
        'excluded': names[i],
        'hr': hr,
        'ci_lower': ci_l,
        'ci_upper': ci_u
    })
    
    ci_str = f"[{ci_l:.3f}, {ci_u:.3f}]"
    print(f"{names[i]:<30} {hr:.3f}        {ci_str}")

# Check if results are robust
hrs = [r['hr'] for r in sensitivity_results]
print(f"\nPooled HR range: {min(hrs):.3f} - {max(hrs):.3f}")
print(f"Original pooled HR: {math.exp(random_result.pooled_effect):.3f}")

## Summary and Conclusions

In [ ]:
print("="*70)
print("META-ANALYSIS SUMMARY")
print("="*70)
print(f"\nStudies included: {len(studies)}")
print(f"Total participants: {sum(sample_sizes):,}")
print(f"\nPooled Hazard Ratio (Random Effects): {math.exp(random_result.pooled_effect):.3f}")
print(f"95% Confidence Interval: [{math.exp(random_result.ci_lower):.3f}, {math.exp(random_result.ci_upper):.3f}]")
print(f"P-value: {random_result.p_value:.6f}")
print(f"\nHeterogeneity: I² = {het['I_squared']:.1f}% ({interpretation})")
print(f"\nPublication bias:")
print(f"  Egger's test p-value: {egger.p_value:.4f}")
print(f"  Begg's test p-value: {begg.p_value:.4f}")
print(f"\nConclusion: Treatment is associated with a {(1-math.exp(random_result.pooled_effect))*100:.0f}% reduction in mortality risk.")